# Non-stationary CartPole (oscillating pole length)

This example uses [NS-Gym](https://github.com/scope-lab-vu/ns_gym) — an external framework for non-stationary MDPs (we use it; we did not build it). mouse-env has **no NS-Gym integration code**: you build the NS-Gym env yourself in an `env_fn` factory and apply a small adapter that flattens NS-Gym's dict observation and exposes `info["ns_params"]`. mouse-env surfaces all info keys verbatim as `info_<key>`, so those values appear in `result[i]["info_ns_params"]`.

Requires the optional `non-stationary` extra: `pip install 'mouse-env[non-stationary]'` (`ns_gym`).

## `env_fn`: build your own env

By default, `EnvConfig` builds a Gymnasium env from `id` via `gym.make`. When you need to construct or wrap the env yourself, pass `env_fn` — a **zero-argument factory** that returns a freshly built Gymnasium env:

```python
def make_my_env():
    env = gym.make("CartPole-v1", max_episode_steps=500)
    return MyWrapper(env)

cfgs = [
    EnvConfig(id="my-cartpole", seed=i, episodes_per_task=100, env_fn=make_my_env)
    for i in range(4)
]
```

Each `EnvConfig` creates one env instance, so the factory is called once per config. The function must return a **new instance every call** — never a shared or pre-built object. Each env needs independent state.

`env_fn` is how you plug in anything mouse-env does not handle natively: non-stationary physics (this notebook), image preprocessing (notebook 04), custom Gymnasium wrappers, or any env that requires construction arguments not expressible via `id` + `kwargs`.

Time-limiting the env (if you want it) is the factory's responsibility — pass `max_episode_steps` to `gym.make` inside the factory.

## Info pass-through

mouse-env forwards every key from the Gymnasium `info` dict as `info_<key>` in `outputs[i]`. The adapter below puts NS-Gym's change-notification fields into `info["ns_params"]`, which then appears as `outputs[i]["info_ns_params"]` so training code can observe how the environment is shifting without accessing the env object directly.

In [ ]:
from typing import Any

import gymnasium as gym
import numpy as np
from ns_gym.schedulers import ContinuousScheduler
from ns_gym.update_functions import OscillatingUpdate
from ns_gym.wrappers import NSClassicControlWrapper

from mouse_envs import EnvConfig, make_env

## Adapter + factory

`NSAdapter` flattens NS-Gym's `{"state": ...}` observation and turns its ground-truth change info into `info["ns_params"]`. The factory builds a `ContinuousScheduler` + `OscillatingUpdate` for the pole `length` and wraps CartPole with `NSClassicControlWrapper`. See the [NS-Gym docs](https://nsgym.io/) for other schedulers and update functions.

In [ ]:
class NSAdapter(gym.Wrapper):
    """Flatten NS-Gym dict obs to its ``state`` and expose changes as ns_params."""

    def __init__(self, env):
        super().__init__(env)
        space = env.observation_space
        if isinstance(space, gym.spaces.Dict) and "state" in space.spaces:
            self.observation_space = space["state"]

    @staticmethod
    def _adapt(obs, info):
        state = obs["state"] if isinstance(obs, dict) and "state" in obs else obs
        env_change = info.get("Ground Truth Env Change", {}) or {}
        delta = info.get("Ground Truth Delta Change", {}) or {}
        ns_params: dict[str, Any] = {}
        for key, flag in env_change.items():
            if not key.startswith("_"):
                ns_params[key] = np.asarray(delta.get(key, 0))
                ns_params[f"{key}_flag"] = np.asarray(flag)
        return state, {"ns_params": ns_params}

    def reset(self, *, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        return self._adapt(obs, info)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        state, info = self._adapt(obs, info)
        return state, reward, terminated, truncated, info


def make_ns_cartpole():
    update_fns = {"length": OscillatingUpdate(ContinuousScheduler(), delta=0.01)}
    base = gym.make("CartPole-v1", max_episode_steps=500)
    ns_env = NSClassicControlWrapper(
        base, update_fns, change_notification=True, delta_change_notification=True
    )
    return NSAdapter(ns_env)

## Build and rollout

In [ ]:
cfgs = [
    EnvConfig(
        id="CartPole-ns",
        seed=42 + i,
        episodes_per_task=100,
        env_fn=make_ns_cartpole,
    )
    for i in range(2)
]
env = make_env(cfgs)

for step in range(500):
    outputs = env.step(env.sample_random_inputs())

    if step % 50 == 0:
        print(f"step={step:3d}  ns_params={outputs[0].get('info_ns_params')}")

In [ ]:
env.close()